# Notebook03 — D200 paired RGB/Event tail-complete export

**Fresh Colab CPU runtime → hardware accelerator None → Run All.**

This replaces the active `03_V2E_Event_Generation.ipynb`. Archive its predecessor as
`03_V2E_Event_Generation_D200_Full_Horizon_SCHEMA2_COMPLETED_ARCHIVE_2026-10-11.ipynb`.
Notebook02 and all historical data, checkpoints and results remain unchanged.

## Why this export changes

The completed D200 coverage audit found that the full-target-horizon filter omitted the last15 conditioning windows of every demo. TRAIN retained64 of2288 available inputs at/after the first positive gripper command; VALIDATION retained14 of575. Both retained zero conditioning windows with recorded cube rise >=2cm. Closing actions were already present as targets of earlier windows: the defect concerns late conditioning coverage.

This is a measured data limitation, not proof of the sole cause of learned-policy failure. A positive gripper command is not verified grasp; recorded height rise is not a scored Lift rollout. This notebook exports data only and makes no model-success claim.


## 1. Configuration, CPU dependencies and Drive

Use **Runtime → Change runtime type → Hardware accelerator → None**. Authorize Drive if Colab requests it. Run All needs no manual cell selection.

The validated D200 source, historical paired exports, completed coverage audit and historical run metadata must remain in the project. Do not rerun Notebook02: existing camera streams and raw Events are reused. CPU-only dependencies are NumPy, h5py and Matplotlib. No Torch, GPU inference, simulator, renderer or V2E emulator is installed or started.

The source dataset is `d200_paired_gpu_v2_93f4f6bf54a9`. Split seed20260927 remains unchanged:160 TRAIN/40 VALIDATION; common validation10 is preserved. Training seeds42/43/44 and final rollout seeds1000–1049 are not consumed.


In [ ]:
import base64,hashlib,importlib.util,json,shutil,subprocess,sys,tempfile,uuid,zlib
from datetime import datetime,timezone
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/DiffusionPolicyEvents')
assert PROJECT_ROOT.is_dir(), 'Project folder is missing.'
missing=[name for name in ['numpy','h5py','matplotlib'] if importlib.util.find_spec(name) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
BUILD_SHA='91f30c3f7c12b3b7ba79a2b4667f8613167ef6c1d0e1a8bd55ea2107b9757302'
WORKSPACE=Path(tempfile.mkdtemp(prefix='nb03-d200-tail-'))
SESSION=PROJECT_ROOT/'results/paired_source_preparation/notebook03_tail/sessions'/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
SESSION.mkdir(parents=True,exist_ok=False)
sys.path.insert(0,str(WORKSPACE))
print('CPU tail export; no training or rendering. Session:',SESSION)


## 2. Explicit endpoint target contract and integrity gates

- Retain every available input control step **t=2..T-1**. Observations(t-1,t) are genuine stored observations; no observation padding or cross-demo samples.
- Predict16 actions beginning at t. When t+k exceeds T-1, repeat the last recorded action from that same demo. Execute indices0:8 in future rollouts; no action-index shift.
- Save `action_source_indices` and Boolean `action_is_recorded` for every target. The latter is provenance, **not a loss mask**. Planned matched training uses unmasked epsilon MSE over all16 targets, including endpoint repeats.
- Repeating the last command is an endpoint continuation convention. It is not a measured post-demo hold and need not be zero motion. The report records terminal arm/gripper commands and padding fractions.
- TRAIN-only normalization retains the historical overlapping-window estimator, now including endpoint repeats. VALIDATION contributes no fitted statistics. RGB uses fixed ImageNet constants; Event channel statistics use TRAIN only.

The generic endpoint repetition mechanism also exists in the [official Diffusion Policy sequence sampler](https://github.com/real-stanford/diffusion_policy/blob/main/diffusion_policy/common/sampler.py). This does not establish that our exact Lift configuration matches theirs.

Raw/split/source identities, every source file hash, native-draw audit rows, raw scalar alignment, all late visual rows, exact historical-prefix parity and independent reconstruction of every Event window are mandatory. New NPZ files are read back before completion pointers are published. Corrupted completed records stop execution; incomplete attempts can be retried in new generations.


In [ ]:
source=zlib.decompress(base64.b64decode('eNqtO2tv20iS3/0reDlgSI0phqSeVqIBjMSzFyCTBIlvFlhBIFpkU+KaryUp2XJu/vtVVXfzJcmbGawxGLFf1dX1rurOq1evfudFGWUpD7ScRQUPhmW2L3yu5dhfVjyFb5YGWrpPeBH5LNb8LK0K5lelpX3KtDyLI/+oQU+URunWevXq1VWU5FlRaX55MLUdK3dxtDG1KDO1f5ZZampZaWrlbl9FsalVUcJNbb+PAlN7jvIwivlVWGQJYFPhOk3C+gJNMRCwiuMqNaLaAtYzHEXtDyjnR42VWppfXX29/bv37X9utaWmT+ehy+Z+yN2J7Qbccd1xaM/sebiZu3w8DZg9YxufO/bcn474zHWCmRv6ztSZu/AxmulX3758/HCvwI0Y4/4m4Jv5aHYzt52ZzdiEjd3JKAj94Gbm3gQAk0/C0XxzMw7Ho9mU3YRs7GxcZ8THY/3q99uPH97f3n/4/EnBnM9HU3s690fj6Zz5/GYzCkebyYzfhOPQnc1nk/Am3Nj2yHXGge2Mx8F0Mh9NNtMJfCN8QPHz/359d6fgcT6b+ZvxbMJuJvMbBjtPR2M40mY2cqZBGHB/wjb8ZjJ1JyEP3RvAbDSaMNt3575tz270q1+/fv7HXY3feB7Y9ngycQEFAufP+cz3Zzb3R3N/47vTzZw5AZ/fuP7ItzmbusyeT0J/ygF5Dmd+9/nT/dfbd/cILYyeQPCqXcH5MCjYYzncFuw45E8gZcNiuxlusn0awJSDCwtvf7v7evsN1q10tuVpdYj4o25qepFtssr2+JF7UertQGj19RXs8eGT97fb+zvv/d1vn8WygCeZ59i4SHzezOvv+QxWXQU8BBFlHsqjgaI4WFxp8LeD9VKiLRh3J1NjQAOPUbXTspynNBux2egDlL1QLMS/MCu0TZz5D1qUalHFCyNmySZgCy20Cs4CY/6zY7tj+t/A1Da6PlhoO2ufo4gbtFJsVvBqX6QwtONPQbTlZQVYEM6oYB58sH1cGQeJdBRqURmlZcVAmQ3QyS90IAWmrAqYqmbC6ViFPXCGCnS7rPRm6sESXfLQoPMl1+6POb8riqwwKvgCUJbnpSzhnieR8lmapWg6aozUAbqUROStAJS2xN1LUGHvgR/L5X2xB+0uec4KVmVFuTR0E/m10AdmTVwWx9kj7Jsuf2VxCfMlFZZtkgwssGdZwI3B4JR2+8o3mqMqs2Kl2aOhLIsFcwZWVGbAyoTVK0sWcs/fRXFgFFlWmQAjZlV04Oq40AmCg2SnCQNgd5nFB8DjjZbDCPW+rlc1w7SclSUHc5Zr/7UUoNAe54CGp1Z4VSYAA1VuiyoKQXHIgmq89FnOSxC3Eo/ESl5ZepsHuTgCEDaJfA+JJSX4wOK9OgCBkgcgbXhDXRZwBBTQSh6CqDBEQ7GLP4GceNkDNcU5Ko52mRVHAEXLUWlIVAiohV/XulUl+VC/Rpdg4f/GBvFKgiiOjT7V8KzHAtTJq/hT1REiPAE4HjAdabV0zR+UEcAdhQSc2VLfV+FcH9Q7ZiWwJo/BIhv15iadRcwJoxT2OC5aqO3TOEofjCQqQQW3DUGI6gJv8JOS5kX2WP5HSQ4KnYLAIFwl2dSPegXQSZeDyK8sdK3YaTyQmYIFaKRwHbUfZGvQcPI/xMTGcCJQkF+05Sl/BKrxpa6fMqNnVAkG4AJEtN7DSf5ekGUNTWAGjwNEp1ziyYCAj0JQdmBrYUrTgcc0iPaXOH2RyS/z9wAhU3j0yjyOqrYbkRrd9TDacqnVgQUQ4iPYewjL3k9sEV29BnmOQIchWtMIouaDk9vyQCo0MpSkP85YUBqN5JBzEdoxgEMTMJhbrnT69NDvlfr6Daq86G92UoNttGNkFq4klLEFVkX2UNfYJhOFIwCpMwvb1OHYHUrUAMCsBVH5zyxKxWSCBPQ2AhRBgTtKJDX73r3jHssOD1pH2kAw0XXrF/nRjcyAKXdPOfcrYEvBMe6ADxm2AJk06AooYBY7SL6QhMvY1vpHlP/a7ACi/NyIco+BzyIo0Lf53hNBefsQ5T5J0L7gAn3QCK46ykoHf1/tgXV4Dl2Gg63zfLn99k0/s6zg/9pjIqA4j8sR2t73eVmG+7g34GdJHvOqu6DPmHPoiSM9ZsUDLzwRAojFrdAVmQ9zGeydU/jiqexDTFVRZB9+L7JQpOzsaeVHvRsHdDf/QZgkXxEcX+RP3mF8HnATQHcCwoL7KMmljEqhmRUBULFr4SQKMHqer18+fPx832Npf10WhmhUvSylH/AdUXXU1z2AL8PId8cy8ksPA7oUDGfICxT4PhSa64PcouzKNS/sdFZhBa+MMIvBWKNzE4pHUrb8BMEY2OQiqzI/iz1kjewjyyjFibqkigs4yqOKFhjDpKtzcuC1nrA0CoF3Ur86JrRtMpLeod59/u3Lx7v7u/feu9tP71HX7nTliTtHgJCcXDOiuGhBIymIAgGus0KB6Zz6Ehgp6fXctn51APQOA7F6gCaGgyEE31LkBfBZLGvC+O+XwC9e2LkJ1XUYO/AU1ZlWtJprE7PBmJc0IL7WfyizLgwTcOy7LvQEKFP6HOLypySGnKDYpxipd/okOtgPjIKw0EpBPU39EcJCCAb7/Q2SZGgQ/wqpARYHrC7MeYa1UtNF6cMrGRrAUo6BPgRoz8SqfRBJEWpBbqZsIhDWowWxS2el6O6a+D+6NPg/IILPkutyH0L2TJYEmmhJVH6MXXIUele6B4m0xNHjBwwbqbX+46qj5BL+2yX55IYJmFp8SJWx12SVqFYT/UoZNIy4ILImy4c7NyAsCLWSEnOsvr9tJVBK3xGMiBQEKNj+m9iTxF6GPgtNv8aZImQGyeiJ05s6eeppVkfTKJ8Cz8ceO3qiSkZyuG1ZxASjqQOBXrbHUS1RJTUOeUZnpJvSrU653ji4FfoWLM6d8XUNAOIlyfHlpXot5sMAk/ThjsXhEKPu4cHRz2MUA2P9ox+/BJZ8QMzZgQdDsWzIkz3kpLyGu8U8YaVvC5bvyAvULNmu9B0rgkfIYbwUIqeIecqXAnggIAbSRHz90+8f3n+41VGctogDqG7RjUlTSouPxoOcI7DhMAss+yNG+00es9Lj+JDkUc7JPoRV6xOxAWHvWUUqLUmdzoE22QZGDhSICUo4c0JUSB/YzMzbZUX0DHFalVUsFpNGUwJ7EBQ5uGCoeFWBmW1T5YCCeYhQeInE4EV1gg0DgrYZSAvnUo7HbhvVAx6cSgcBcC1JokpBYf50NB+Px5PJjLnueMOmnE9898ad2jfONHQmrm/bm0DvQcuz0sMAV7o36En5tt1jdfbHOP2weqARu03wMtomTK4zdX9fQTDi7dAWgfQ8eAUSlprlDgidZlHJW50FD1H+suKI1I8ysPo1jy7475eN6WV/zlb6jykgHY4fxfEQWulRJoyVyxKcJPcgJUxY5e/oADie7asSEgQ0xB5VNOsRWinESxi27oiSKTnWig1rrAEVQfXGTjIfbHkZUb4YMjCzZC1hYu+8cbZFr94Wao82ry0lUB/MB0k+OBleMH197fSgpLIQ9ZjJOR5VcqXk//zyPu0oMJFFwx33H1Teg6A8ytKle1B+RwZ3TRGhHd29vuBpm+qCLCYg5CXVQlQV4asoEYRd4cAklsoEy6VC4CXi9AopJ1zDFBcGatpJggH00QmDQbFw6sN6udTRLuod7SLJ2afSKXqc+TuChvrFyoq4ooSIqurtgUZiT4ZQUpG2USoFdt3JsBGRFrcoJCKXIMugsuWJ4pYsP3p41fMA0VddC/bPFS5aSy9rK+UZDVwhbE2b7CbNwf3EKH41/edjd72bMZwp7/qo87L2GkDARMap5zfq6OZCTlEnSqq/CZU1/W8QwRai1qOGO4FPt4irgiYAJ/ghgwKelllhiGgPUqOKFRUW1IJu2UNFg0GUIE4ukUd2Ako5XznC17QHKI7A3jS3QmBcNRKeIM5MbRcB1drzLLoZoP2xvHoygih1FC3T3gIU6vJ3WcnTGuBKHme1MO219ssS5g60n7ROL64drGWlEkmgEZrPvMjARYzM+Rj+A0QIg2VzgE7NFJVd7N1cCghoNOvJ1LCOLWbAts7aAq3BwwA8kN0pblCPuqejnegFq/mQcEGArPa0UOVF9cswnvCg9uAn+HirAeo/GUfVcxQ9cn4PKCtAtz2I6SFpazBdjNYt1LB5gpzYmNCK0mYpLlwNHThsez9KPkDBE1YKg2TYZmvJL/bANJx2z1t70PJhPliWquywSDLohD9v6FBBYDEw1bTMNI4r3HdtPonfgelYg9YVATJs5e/WAj74ICeXS8UsDFbLSoyGeziT2nxIxAh7SABtCImEPUXJPkFEBARAxHwaNKe0pTGgkp0AX7NYrMBSrEDPNWmWxBGSfmNo7KKhmLYSY4PXlj3pWF8pjCf67uHNGQ8uq/2rV6++8RhsFjBZI5uvSQJrdV6gkfnFgjDaf4hcNHARKd7wa9WOa0gpvDjkLKEb/47mNwyrVb7V9ZKuh1FBvEDPCEtKzgp/1zkOUdakvTCWWkL8GFaq/ol+60dW76JzqyVZz5jOFeG1QPDrDj2vZMVKlGW5LAHWOSwllOibS1mcwg5RSWj3PCWxbCFZZXKubj3/fDnrfP2sA+g0IaUIihSEBbXDulwCETdpeeQ/xFxcpp0Us+ncpiaPS8W6HLX8WcYrJfq5Z6zt0gTZEpmWjFVkYEiKjAmsJxESUBR2TQlHzkdnQfcNtM2gTq1EW7g0St9TE2yKuGAQSHTGZmKI8O4MOPOLxltu2TLe7VG5y6Vh2qltWGEQLQA+qxF7eTDmx2Bh5EZCnsnQY1RgpIPXrm1CihgvHT6cmQV+2ReBtbxDh75mC/q/xYSwRhx8b1WvcxYOINY0Zwt3ul7XmE0FZvhV+9xGWbo1zZfxlmLWLO7AkwT/MwCVvCJRSxowWpBOXVJnP9Dkzl61+HafYMip5uaICC9qIyB/rYCLRwr/5gZCTT/7pkGmbu1imKxwYXEU19XhZn+bVhJ9UmTtxOIUwQokxOuYaLtTfrZlRi6UYM/YEIIChrU8A+VCwfYSlJe5jJi29MZIh87gZ8e+dk7dPWiUbf0ITMS7BdM107PArH5x6UwCqW5MEc2BqiwJxyTIIFOWdOjWdadzNWk1y5ldNvIvFLMvWnktZ5juLb8/LJ6x3FPnos8W1XpFTbmseC452dBk6EwasrQiYXrp873lcmAtlopOjbwpXYYHfnjLK+xQuW2Ot7LZvt3l7wt8IyHr5/TuQdXOdU+/TrKAfl+uo8MsWejZbgC28ND6+qTG3tpf7dutr2NNXdCOlAfR+QHJEitWPeKgfYXfwV8A0HaZAkojmD8GqccCckIA0X8wVtJUrqpFde1MBZHoJoA2Wg/+zDYnjF2b5G9WBGsIecibPwmp5ksbkPRyMqgHkbWtNzLfFAXcpd1kOh3x6D1k6SmXIcRM3bwMXtAoYQWWz8oenJpl0d+KRGoLIRJacyRMhZzXTtD3oGjzzrWzkFpTk5JrtETXkLRFgwiZWy3JQlCws3c5/UMMqClRK9vaFJiu4LNxdQGP5Ql+hLKti6yXiMsPSFiltm80dVjsbUTrjSaPit21nPTZwA8NadXapkcuu1ge6QKS1ZWFuE0am6cxIT9cjhbF3pfGJSbteLJ9z6Mwr8VJYd6VL3RqA1OVTE4iBQXmF4gy2xULCazfraZTReTt0jm3pjfWDqEIIfWGtIcKOhmYDz9BFIYwTUSvvyyHGAkPZOmKOq0kSg05Mu0OsCdj8HaJSrUC43KNE+rN2vbgeilxaaPXyt66KP639luUgtKJZyz4Zqi+jsP4qtDowgMjR3y1HHPx1GkHbbpA1vClkZal8dHqgf1WZQW93AcVq0rxWoaCrdfiqgjLh5C753sslSb4dCpKkn3FNjHvgiI06GqdH6gAdoma9kBcaUIPufUt5tdq3MTi6hLTO9hAH6w7W6DViUyQJzwcp39OgM+aScC65GqexoSmSLZRpPGtA1qqWoaiNUIbWo4tfu0JWCslRGrQnuBv32Kpv4Jv9lFcLc+VTyRJOhiAt+H40nlJKm2IOhD+sE1pALpDCXDQepDV/pMOZomr5LdJAJsklZpvMTWa0I17wHNOD1i1O0QSS7zgXKti77cudiz9nHGn968n5r22G23TrozFX7fsUjsa4067Sz/tdu8NElMsNb/Xl/CL9oW8qav4dSHyeAxzzF4AvGjMU/O04jSWXjRektIfiecmq3ZyCoBqq7apRw3RxXQPeOUBiwFLybTWY44XroYXzlwFq2V9/4IuUO7H4mibJsiAhXgN9Yeq5ol3hOqlYslLvIWUjyIEi2Tfkuo/sgFSFAC7lrL5un4ZgS9VlzjUeq1Lzb/0Wrf/lpEeyOK7XXwdUki86kkfvnjv7379eHt/975fI1LymlN0KNRO4m4V2zjbGPrP+uCMdcBXUPj+na5XZJkGX8fTu2TxOl6G5ZiKU2huAUIQlssXMJQ7XjAKz+JRsJGb+C8jcqv9yF4RenD5pTDStftQ+MV3wi21wJVX/w+U+5ZE')).decode()
assert hashlib.sha256(source.encode()).hexdigest()=='ad428b80b3a28d9a427846d06b494f31d2b8264a5fc10084a7f09065294463eb'
(WORKSPACE/'common.py').write_text(source,encoding='utf8')
shutil.copyfile(WORKSPACE/'common.py',SESSION/'common.py')
source=zlib.decompress(base64.b64decode('eNq1Ov1v28ixv/uvYPOAUuzRyvKbVKoCvsSHC14uDmxfCzzDIPaLFs8UqZKUP+L6f38zuyRFWnIuRXsJYJG7s/M9s7OzfPPmzekdLba0zavyuCqLR2Mj6yZvWllyaRvygfLWaKptzaWRC1m2eZvLxqClMDY0r6UwhLyTRbVZw5zBtuJGtvM3b94c5etNVbcGb+5sY0WbVZEz2/itqUrbqBrbaFbbNi9so83XQGa7zYVtfM03WV7Io6yu1oC9xTVGh+cLvOoJQVuJq/qZ/l3j+lqVsqddbtebR4M2Rrk5Ovr76fnFx7PPxtIwy6qVrKpuSXSsZTgu73KR02PhEnLMZNMeZ3lJi+M797hpabttzKMPJ5cnF6eX6ccPiAEBU702vdls0zs3TbzMz0KWBT5NNHj688nFzwgdJW4QceEznyTCd9wgC2NXyjhwIkJCEQacehksFFQmMfEolSRkfkykK73Q9Zh59Pns/JcBXRhQzwlDl1M/IE4S+9zJfCIi1w1pRHggAhmwJArcyImzxOGudDI3kiFQ8YWTeObR5fnJx8/p+7MPpxN1+Mck6DUCUml13Dlj+PTi5xNcEzAuHB6GIXcSKYjjZ8IVlMUeDygFiWgSOIGIPNAKD7gbyiTxAxbBswzdzHfNo/OTf/TIwjhzacwz6QbEFdJxXT8jEYkzBmryQ0FJRBmXDol56MnIdUDWjDuhE7vw4EXm0cWXTx8ve3SoQM6EBG4i0CcomdKA+m7giYyLJHITAThlkHkxS/zM96KQJhn1HeY6nvR9QHf26/n7QVYpo4gzP0Kh4oQCVOiBzkMWgRlEJiTIzGQShGBYmbkJYPG8gBLuxpyQCLzhp/Oz/zv93OPzY0GIHwQu8VyFjscy4jwiknsxZ6AtFlNHyDhxuceJpKFLSRxkPJRxFEjg7/3JL6fnJxeA7MqkNxB4d7m8N23DrCtWtSSVjzLNy3QFUWped9Ig9JNZ3zBzgb/bMs0FPGpXhuGxO2f5gxTgXCS9c9JGwrObRjT2QtBRJESEpHhVZvlNiqGNaMbTISgXDMWiUILfJTIMJQsZqJlHPgVnj5gvIB7ciIZ+QsEAjssRJUae4g6yzwYePOAAhZIFpKc7iWQQ5C1Op4TgdArmTcATkjDwXAeChwSRlzLOM0KybM5vNy1iblbUDUJE4Mg4lJmgPnMpS2QGoRIyF4IjcYQXMUeCQ1GIFEATeZ6kGaGxEA71IFppQIT5DOiAHzlm1N9ntJbNdi17Vv0xq3ES+uAW4D3ET30wOXE4O8BqFGcQBgkjcchZIjKIZScSJGHccwlzGM1c4TGeRFEgCKYe4BUiJvYIYx71gVXkFbJz2TaHjK5nfs/uzE8yT7g+9zz/gN3H0yISEbz5nLhg9zjyXO4AOi9gPmeUMaRKXEdClNFEgC58Jg/Z3SfftDtOd8r0PALi+r7nR3HipCJwCDhTdECZVEAuRTWGMckkYICkSLPAY8CKlCyTTuRnkHBkKB2HOzwIRBhlmfQ9TDyZ8x/b3QOjxeCdJI49H1yUeXFI2CEXzagHDghqCmBVRgMX8jmkAcyFURx6CYsDEmXwJxRRHEHCIX7i+6hyL0s8l4Ldn4+OhMyMbctn1sKoZbuty2GjnJfV/azfK+cAY83zpsqqek3bmaUWAjMpbsUz3IWtxZEB/1aQQLqNfK6ZBWCcuM/blVFtZKmgbbNmpoW7bqbX4T9AbrCi4rdGXhp5K+tZQddM0EU2ryUVs/gvDnF99ceymWkC06v5doMcz9Q6TakTZDVfyQeR34BPdPxiXZHCA90W7QzrGdkxnWdG3uQl7OJQzugZ+4uSqUfWtHW34tUFs3IzLwWta/powyMkXFnn3NrhUGDztipyxZFileaNNC4fN/K0rqt61sJTR2eepiVdyzTVvHNaVmXOaTFhvBd1qnCUcy6grGk61hqoc9Jb+dgsL2t8lRta07aqm+XMtOH/wrTswQq0KKp7oF0uf6JFI+1OYcux9qw5FH6VkDPL2tMyIF7nPEVwbeoxwziwRNVqn3mHf+fADeSY+fpW5PVMv3ScygfQVVrdqjetslZi2Ubrx6Vai26lFKUQzvHpB3PerjfH5g9YL87xjz9TbHYI6sedyw3Y5vc1eFzayod2X395iTXt0rW/Sze20k1e3izNbZvFpjVQqxpw5E1BwWcGwrZShAJR5WTxuBhxtS2LvLydrfMG3O1mpwkVfjSTKV/lhZjVVdXafYLpfQPGtKrxyQLKTVXcgcne9YDdbL9MraJNI7Eorlqjn4C4TymDxVuIM0sV9eZ8bioYCNQBDCzXQs1u/loiZwa85RmeC5RhzJ35FT9vB7IjxjriCPWnJUJ1JwhYDzz0K9K20iKNQwCBlFa0HeFAoZ2vru6b/5rvqRBS4Sty3s7xoIFDs1uVuoCU0gdQVO+33Zv1RzjuLp3uPMmE6q6U9+Axcmmae274ItsqLKisegnamn8Aif6hXmeZneWyEMhSs0QBrXcaUAfJCnIxQE0HUeqZUvYf5O0QiTlm+lSfHmfVBg+jvW07z+kGr8w1fUihBmnrqkhxl23M6+XSI0Q51AAlHyTf4nO6quocNjqEig8ibLacy6YBwELsMDovEMIBUVYwXcMvzruHcXHYGyBdrCuEMfEhdYiOEKynmuWOQzhz1lWJx2ZVagHVdxvYQfjjDka/D9NYfuwm8S09gGXMWAFepIat5RKfNUJrEE/Ptx1MB4QDHaCCnCBEFpZaFB0vNS1v5AwrO9slDrF0GukkGYN4COIpEEwlsDu2K2mokhMcpTzUTlB0VMRJylcGX0molqocotocM7WTAJKJyJvfEKQXoUslAHgnxRJBNTsOsuOQANj515Pv2r5n+z6I4IYkcaPnlylzpKZ/jRX05x41yPQT+r0hh37K2xaqgBL8/W2zKfJeHMhExqaGMzdvpZgKsvMj1XBJNRHlSuh5vDWnTtnQ9aaQtQJQtFIlW/rQuVyXQTHUVKTJDRR6QmLBMFtXAuKufbSh1LnBykGKacT1AJjr+iMkUlfwOHilC3dbl8XXY4qZ+dQvf06f1Ipn5Z/pE5pGEXs29V4HVhVbKDWnYf9aUtD5HvWSQwrDZD37mm9m3wor+5V4sibbzNWTCWoxF7+nI9vsR83FMG8qAHOh4SaJ2NhjyVwMOrDHLOnxXrYrHLq2nifIMBS+qf8JICBAuG8qBwtehRFcSKEzJJQ/xu9mGWuP3NhfrlSjwe4Pn9fX00SPB9Fa8qqG+kb92P3ivmzXncbHZed4V/38JLvptVf9uRaioF+3G1Mu2wOOz64T6MnEIRKDzWHZIOgYNciFRyaIvnpTgx0RcGgCHsLYHbt1CL+6ckKjxCNakX+lemPrpBh6g4eoIFuQrjr17JqYh2Cx7E/vsPurt8tRl3CqRADrTqsTKOxvjfHuDjZPapG5OEzJnqBcHKSzi6nOVmM49Q5oUFhzMejOnirMXAyaerZeNT6k8csuaxt6YnBG2H8wu/Y5m2c3y5c8TMTPbmBi2LJSTHRriioLlT7V/LYE44h8rWqOK8eNbRDXDhx3HxV2omFLkHVaMdxytFBYqsQ7fKOpceXj7mHbgPNB8nwB54xYo3oWmMOZaDfB62qTNvlXVQdF4R5qkWfZFk07rqbIbj1WcEy2ShXzJEl2MyoLwqi/zy8uGtBhW2W3qqjTfhOZUBzDSFoXkGVb4FyWlBVIBw76hjrs7RGTD3h/MLKYu8P0PRHVITlUgo6ExZog7UXuC4/xlgRg07y586ZZQ7HmeJE99T7V5VD0z+/JuO9eT7ZXCt/EFRXdccp9mYR1jaChXuTbg2n4APKbomJUV/fTNK1HxhRGQXUA0atZu5v/HluOQF9m6W7q+zL5AZzdwn8jrQ96/WamHkP9e4l6YjfY5JfL3Rz4k67t98CnaRZXfYv527xU6p6pegMq1vsUTJhmG881X6lGzGxbFKlurZqTboaJ1seKQyE/WPQOMdBXdtjYbHRjBevwisNJErVu/1Yx7CpMy2AYwGw5gkPmJwNHfQkENaOufrCSHBWKfX1o79eC0wrwejEmC1iAFHClnvaY6q4msfh/f/bLl0+nl6cfzD2o8Q40+EIf6Yh7FCXXekAzC299U/r6m1i7MP1unAp+D2PfVGr6mg0Ha1kK2PBuVIELcn45ubgwd9P3Mr9ZtU26LbutWSd1ZXrEf7OltVhq9Pyf27zJVYip4SkLamii0h2pbq5vOgBLSvBGp2OdvjuDqM3nAF443zZVrRfoHewv4xVjMrjRUdak3RLcTGWNl/Hm9V+XjjwOFHoGeUt3cZ5U36u0VJ9JeWHZdacmOp0caJ9MKHDaXOVYUO283KCPwgnyNziUVvUjDHxVTgvczXlzB88geVFt2/l64+ObtgusZjmaOYXIXGOfB9ulB+Y7JHtGeLlOnRdSCVAQtSj0gYV4YZDSrchbNfn81+WgjCEQ8c3WvetDypjnrVw3M2vXMlMdxJdtV8SyO+eMWpjYs1TXI9aujYmanFnwo0qjv+naY3qPslxqlvbSlM5Tss6zHPaDLk+9kqH6lie6oKxHbeC3ZrewMd/OVMxhFrr+wdTKHa43sJvRrZ6rTmgzuiP6jJ9P7LUkQX+qET4vKiqaWb9aWUL11a2dmoDCJDv9aZyd9ulMzvrfnaF3nGl0MKaG5AOXm9aYnV2oWxf779jk148nynzgQPr1f+WjfhiuaMbegLv3zPxY8gr39Fa+hcKl3gJm2mJPszXu86IwmET6cFiAc7Y+IUERtDDtnfLtrNg2q1GfecSzUgEafrNlRd6spnbvCNlFBRvtK7tSr2K95+y0bJs/nXz8pB5Ofjw7x6GuO/MiDpZPz0PIoIeqjRQqXylmyq8UcQujpcYmujVSUe9Ik2hY4JGpzcvtzrYtrW9kd1vRyWS9Hfri7/RnQHNebR6HOLH1mp3CdNQsp8HUXykMo92qPsjgIHeKTVtIIuDMtGwgjRoZBcj+AHdII1d4HagxzfduJazrcQCPb8Om4qlL4KLtktrgr/9jnOH3Vb1P2Ua3fxiy5xMqK3AqANC9UVXw4GGlXwPeO4Tu0aFgm5QCB+7CUnDN5dPtQlUX17u7jP+gYnn+rjA+VEXZE7amcQ2aEArZawlOITyQ4F5aZkA0xd85z+Gc+/vMWlhmoP8PeWyUiFRM69KeV1soFNTtiZ3lNXhlA+YthQ1bKZq5D+rlU73fnYNqsNbXT8PlE9r7FWvrDASHp7raqnpM0dPWYX8Efi2JJgCyrKty2eUO7I9T68/4wyzrnVaC7tWyql2l3Y3LkmidpPjZIbxohP2bgsR4XRLr6GVLU9PbOTi9s9ndklVVMaO6cXrVafjastUwezm8czXF3pU5Zk0dROidkpnd9YeQgdl+Wo+P+FYTw4JBBHDSpTN2kyeNzFxop+hQmIvOO8yOS3PRPdjDIVN1XxdaftsszQXe2OhXCxtpW/Wxj/59Pjr6gB/6TL+O892YeJ4bOplHGKdeQrhDfBYlUcy48OLI8TwaZVIwngWMhr6kMqRBSIIkjGNhdjj1J3Jfzs8uz96ffRo+Thx/BumHCZMelb70HemKJBaCRk4ckMx3qM9pHHoyDoPMCVxKAhdAmdeh/3J++uXk/OTy49nn9MdfP35SX10SwShxncRPoPDwaOJwn/uEZQmUyC6R+KlUGJHQoTLiLnF5ID0HOCYJd2MKDJhH/w+lcCiz')).decode()
assert hashlib.sha256(source.encode()).hexdigest()=='502459c5ba635b432af6d0d5fc06a5c46917324e84993bdaf55a80e057f03483'
(WORKSPACE/'contracts.py').write_text(source,encoding='utf8')
shutil.copyfile(WORKSPACE/'contracts.py',SESSION/'contracts.py')
source=zlib.decompress(base64.b64decode('eNrFfWtz2ziW6Pf8Cm5P3aUYU9QjsZNQy65yJ+5p16STlOPMnVm1ikVJlMwxRbJJyraS9n/f8wBA8CHZzvTe25W2RAI4AA7OGzjQDz/8cBEGy36axDvj7acvRrBdRqWRrowsiPJwabwbD4dGlGTbclAG+TosjUV6E+bBOpwYSWpkaRwtdkaaG9nVrogWhfPDDz88izZZmkPN4sa+CoqrOJrb/yrSxE4LO4uDcpXmG7vYFXYZbUK7zINFOA8W1/Z2Gy3tr1G2iuJQwrjbxE5Y5mHonMXhJkzKS/huBIVxdvlslacbYxmUIcIxRAP5TMC/pknI1bKgxHHIWp/gUXaRbDfZDkEmGdddpAkOqixk7d6708vTz2eX/vk7G7/6v5x+/sX+8PHiV/52cfp//c+/nNqILP/zp/fnl/T4zID/+N3HLxdvz/xPFx8vP779+J5bUcmni7NPpxenl+cfP/g/fTl//84WdRHAzxcf//vsA31tQOVKn+2bII5wxn6+Tfw8XKT50l4ESZpEiyC2i6vAR2TaRbAK/cVVFC/toEw30cKn9bjNI2iKy7QtF9azZ6df3kEnfz+7+AzD8cwkLcN5ml4PX/WXMNZ+GURxXy5/f5Ft+zcj89nZPz6dXSBqPBNr+eFdFualP8+DZHHlEy7TuPBvRv7Llyfz8XI0fzNeVO1qOPFMvc7LxavFKFysVqs3L1++eBOEw8Vw+HL4Inw1Gr18s3x18jpcjOfB8cv5m9UyfDF/MV8em8/eHQ95MPDJROyvsy32Px6/mo9fHQ+h+pDrNTrXK8xfvB4vg+NXy9fL12+OX78ercLjF8cn85fjkzfHJ6uXx8uT0Wr+Jhy9gS9vFuHr4+OF+QyI5PL8w18/e9/MDHqOFmWUJv5VmkdAiqY7OrHNVZQXpR/cADKDeRz64Q1QtU8sZrpjXl4zS4uojG5gcdLNJkiWfnmVh8VVGi9Nd+gMbbPYFrAcCUyuVbUow6ww3RcCFFMFVFxs50AnURH6WZ7e7fyN6TrD8f2zX88uT4mq/3b2z8/e1BRLJuDY6hkZCjrd0MuAZ8YyAV8AzCyPUoAd3kTpVn+12OY5zNGcPXu2DFdGnAZLor8eMqXl5mG5zRMD3zhYVvSQO7nQyUE6+WV4V/Ysi5sn2Vf/Ct6GOVWxr8Od5dJUQfScJ0UWLkojSIwPn/5pBHke7AyubdxG5VW6BQkBCNnAMIsiStZGeRUaWXQXxgNaCBAUOxwEyTEEiq0MIZSc/46yn+GTx4YiI8iBq25CW3w6aRYmPRjRkekk2c6kOgWIr2DDQ8T/gH8KwJ2XZA5IJAelYVDyRDfBOlr0uIGlGkQr1cbrjeyh5QJjZ6ENLUFOJfay3GVhFziavkCWP/KHLchhXIM9/l7Y407YRejmAdCbcVoUIBKgj7M8T/Oe+SUpthlKVlAwRbrNF2F9pcSIXMM8Aqg98WhVsAMCCAoIVhMH6ICaSef/4pVf0nsxAdVEUBnPjhoxORFXxMEOKKMHysZG2oAlXlY0dbpYhBnoRFSRSC0g4EDpgZZMgP/6eTpPy/6/0ghpJ94WBLC/Ai3F7xi0Iqc8TUvv7NJBRQNzAwrEXq3JbZrHSw9LnVWULHsmvZinSyCiZ4II6JURFcYHUGuoc1X1II57pjMYRMki3i5DUzHVNxM4tgRudM0vH07/fnr+/vSn92fAm7BWBUokE2D1YUaANlgM1alj3lOvNIfCmyaITyMBS4DrOKA48p6Fo0qcMlhjwdSk2gBczd6czQgM6oQFrLY3XZmEsKFP5d+ie5MgRwgAlmsdAoG/tmZHU3OdRxlokiGIrPVVCXoMCnNuNoI+DhSPTe41vfZiYEeeg+V5oyFRxzRxQGb1zCTYAKaqiXG9qftmNvM8OWIJKL2mtohomrDnidkywTFEpCoY2hWOBSATTvgBBDYo6tCcdXVnNToBnRvLUdML8TDtj2ai7wrDSAdacTUOS1QzHwRvIsma2pgUIEKRbZoWCOZbXPAGKMksovbv2OnopKvohoqOJSkjh6bXj6fSM8lz//j1/UCCNnQOM5bRagWCgszS9RakO1TAicFIQP9FCVhJkqrb3X46/fwZCResUx9fhqwtv4LKHtvmPCgirHWJduYR8b1RgIEVIAsCNRu/gxKGAurubvd18PsWQOQJCC0Yzu+jk0FyMzp2TFtJI7NYgKoAkBdCO3NblFgFjAEaGqipDdLUJHpoXqiKQfPaxvtoVRrFFiRTUWjWN8wMDGziXhZuwlgDxQIalXV2YdcUvM0T9VCm2EVYlkCvhSftGCEFRUvUA0FBoloCsyYERS+pgWeKEfJatHGSZbTxvDFzlHhHgnk6AtZ7xSyVOVEBXA2CRvXlIPvVIFIfAt6oalds5yTie1xO320sScoQxASDuCTpoObxiydnP+2y3WYw0SAvtUqHDLmG2EPssHQjIPZl/5ejUaVbYVgnL3lQ8FSClQU9ODxFoWhDIKmYp2NLsHaY5/6mWHvIHqxNa7gXLGGQU8OaK8wXETLGahvHfTE15e84Zg23iB1ewx+HErUk/+BjXijE/Zc3OhqF/RNuu0i3qDCgztcwT4veZecsUQiWKHAIvsuNplChV1oufxz9MjvyRlRbWrie6HLq2iezH7XV2m8szybR8g47R84GtwgH1ZP1xVhwGT3sFKpOhzNSajh3eLTIiiF9S3XJxrn2hhNlfnuqjNSYneGsQnApge+AclVXlfknYPDH0Qh7y7iboW7xqR6Uwscl4FY/alT4oCMwq7qm7tXIo/5ToBzxWihyb9O0NbkGO6m99KhvuGxKsGbeZb5lnIGrumT/dROROe4JIqgoRHUo1QYBKoVhgUKSlmBCjo2SReEmK3e9YSfxRcLo+4qYRYjYvkKSkIe6QKM3AtYKvIPSmgge4SIWXmBAX9pWS3hxFV12qYFzUZ8/gPQak9BpFkv0ZT/g1Ql7gl2zaj23SfT7NiRCB9jFdqOI8z97zIA/Di1hYwP7bOMSnNgyLYPYF3xnupfg1AWLa+iWxBy8qYSE9GyFJ4v9UAGylG3GQVcRGBqWphQZgBqxMPEkJZoulct+9pOs6aoyDTiP2AcFn+Z+sAL97Nf744EJ3ND4fvSohkUigb7qJFPJhnYvaNovynj32I6e3E+ldp42L9XuO+ZWW3ofHWMN9ezjV0vQQWeC/S3r0Dy+A6yCUQedB7d7gyMIRpYRMGhpdlfGiEaBYwgKGe4QvGS63Tymo6ylmrhFuuCZgf3GgxEqUNafqUGB2Higkaxa9ZpuohKsAXBry7DGqS15Cx3UKtdjP26rfotZ98ohhMCjrISarl2rt1YnsSnAGky10DCylgQ/TKgd4Hh0pE6qwUjEd2uJx7Hi4Q4F9VZdglhHBfqkPjfBXRfqNyAjUT8RdAcqiZmQL/EQTOmSfBI0aAialQ3XeVBkE0N2qzsoogZQTgyu2ED3ThzjkpnGQF8kDjLphGE4MQYZ7U1ndZuQjYpLzWSSVZ0A9EGy7H2rxSiBDW0Tq/hAseVgPLRZCmumOVrAUDfFYPBtlCzTW6wqEG/X7KMa8ZguOcvC5Ggyi15FAlPxJ9WmAX2vciMwSqZRb20FSIsrzeASzOAm/BsMtIFrur/NGCE/IIba436I2LDJYyltH39LVBIRaOLhINh7S3fp2XCxJcGwF5yH4AXhJP012eR5ipFCQV8UkBOvBia3LwaLOC1wmdM085fg2MVphptP5kDtdrBnkqdlukhjr4ppI7yBKQscfGeCk7yF1ch3rXpis0SRM1cTrXRnTG3q9CRoy/M6t1A8AWSqBuHjJpxwSaXfLOuICAi43ebbj79+en92efaOY1qqCoZb4hCpXm7mYPVxJzgxn8o2E1Ig5DZ1wOB0gomcrH1og6HlEPw1XOSfA1hcJRPy9JY8xWo4chBuMx4MVQ/NiIubQ+M+lUsiu8XQl72M1mHB9Ihtgb+jFW4MmjMMgG6KnuVKDIittl611UZLbCMcC9aKQR0OPwns4V5eBJyraM1uLKXbtfCaJFcocsf2oSXB4jgMcjaicSe3KvSDJXCn6Q7vJRMVaQxyi+P1kolsMA4C8Ep8mqvYgFQUyguU5tE6SjwuhAUgAD6/BQd9GW5SVYgPOPeZ9NS42tS8BqlNa0p7jOswQf86wthgkwhkC9GPHGBEzauNXKKIRt0Gw0D1fZu3ezvN4ghwfRWMj08qAHLvdkLCRkfZM7VTsg+imjjuq4W3sFK4salEGlO2HPjU5MJtEZqzFiucXp5jWPVpCKPd1Mchq7mh+lgkKfxU3jHiSeMkSW0mDm4AT0Sjy4G2v8ujGphHPOLWXlElQB+Q1dahmaTx0vuWa3Tq5iykUEY0AQtERQkGBlMl1rk1LMv9FL/MamJnFYW4x4M7B6C+o1VYKEzZZmWO83ZHlvN+S0VRYBrPpEiCsU4JHKBY8Bc/ttYFKpoVSwH9YAgaHEdoKNess1jI6HgZ5l5T7h1saIm4HsZj85ZSZPQM8GOgtE+HThTtD4l8VaVO1QdIvrMDtdwgx+FrZ53mclVYbxd9v3RT3dWbiInM4SVQRmPuDYkrp78XdDeZafPZU2EvwDp9VnDq7/fLwiZtKcCSovwlyIEF8hjQlucxSdYMFKmeuWiglkSQ1UNrJd9qtLoH0iTz5BuywID3A5TTNcKqanSRrv/29MO7c1BVZ7yQVe1DdNgqZSKYUhykoklxVkkWNmVxt4zeM/aGIuiA0qkqaltCMVi0uNkbZFfRAhAxvQ53AOEmiLchSUR4tvkJpGKl6KoW0hSzukf5b1FzS3l82zdr92G82Do9uBkGMmO0wqqW/GJ2D/T44ASe6ZYqa4zKK8ba7PI5SfaVdlKFu1UsQLo4d5sYXrIrrB3soUMrtpxA433b3m5yFdm6+tDFfKZYMNNtX25gy5psYkpx0OGl0Zx75t/DPFrt8NhOtNlsS9oWAPKWLZmo+eCiOFcSLUH3gksdFo7jmPYq3hZXtP9gdYoH5QiSoQGgB3G0Kv3syofvztVydYyyQp73My9E3wYStrG4wsjF0mELq8gqtxIWZhWti4H0eAYktPP1XGwa0nsfqJyFyKTlVGRW05YUfQCXajIJ6k0IlkclwsXyhb0xgQ5kgTg4iGpZlDa3/qgpdEsR/rAUz7TBz9uBWABgtCr4BI8vubxq5ETFMipot57q1FDP4xEkJ8ROGJIUGw/HJ8M341esLpNleKfNtYVcZRYKbVjo5iEhXDmceIKpy8umPg7ZE6LCQadCAllchZugcpX32bSiulJlWA+0WIsGhLHKUxIDUNqrMa5uJYal7UhFF8DJYwMbD4Yo9npQHvlB+4VlTbcpqS+OgtHS8Mj7sLp9BsKnVG9eNH2i+Rbw3XTKWgdv93YYgdUsaUr3WtQ53UZ3qzz9Gib+VRiD411oLarjvN19HVDUqs5hB1OHK9CL9bFiAwS+oi8EX9TtIvsOkBVha433sQSfakjzTctD4HZYBLLoK8sipPQu4sdaliSZepMuguly9gjEHhjShRVs1F2oznvXJGUdfkPieh49sqrOvANmBO21NNmHgjcpBm8Y5fJ4MAV5uAt3+u3avWFDyb6hSJW0h9DBuKaIKRkHwbxI422p28p2pwF9X3m0Yml5MjOt8xqGTLc6Ci+oy6WP+5p8AAR46vS8XAdN/DTtM7EYe2sIrSCca++R3veELaFivxuvWlSOel1HqOJDeqKq9F2xpXZnBxlM09tifkIttycjCmpLa6GuHdbUMijvClINV3vI497zlMY/Uloe627SJdBKubOFMbbDtiKDQcVRq9jPNqkMJxVxHQAHLK7J8SvMgYQ4kBBBeHLQlIycZkqEHvLZJgOqy1aDIApbAnxc2Ahh1DigCh3VhYTclZDiToVGRai0sIlP2PyN+Ow6yQ5lHbC/W29lC6kgnlBtrPN0mwkkdgdUJw2T29sT0O0M5BJcEVz47nCNaH+IbVSVg8QuvEIWa7UYgt2IddiPDqKpqJkcAruhAo34IE/0pXnZUmW1BdkTlphsNMxxg1bwoNMn2VOXzijrvXYbfdIfbuvtzQN27KY7orB5hKGwUWLvKVExBZ6b/hmO++axwacWNjtiUxVMVE24E42+G3tw2CwpynxLTncj+E7bO038BMgoPoiQdUJbi1hROPB8LNIYO85lf3QyMdI5tLohSVP0yv7ILq2JON9VGGpn1y2PRiezCZ7yXICeN4bua/NgnFVoVp74/27AtUFah4KmT6HqdjS0NqfmkPZHz54aYWT4jF06L+B9u2/pOxZQ4OXj1h5SSW3PEmMinqx6ZKoNOKKLgsI2kz2iQMRZ6pjSQFXI3HTFYFRaE6b1AE3oYO0gjtNbP4sW13Ho0U4sJTF9dXmeqpuZB4bn1+n1zFmk2a7HJzavccq1fLL7psSulbqHjzuLLgmFHBi05SuJT36rDkJ/4uDPJiwDisvwCWhMJMI0sWetmcsg7v7I2V58aAd6oU3hfWW2QFwLlEzksXkokkcoVRkmx1EJZRD4IldOlVfQhTDUk/CwWUdWXq0tOZM8Mu0oq7IVqsFWZ2TpNMfJSzCRoVbHkX+tsZpOo/WL8eOOsWPXdjVEa3K4gcxc0AbWeYZfZQUgUO1ULgxL+tGZ/n70mt9fenQuVk3QCcoyL8AR3G4kQ8pNK5l6pp3wFXPgE76t87/Up378t2OqcUwHTgSkqWsPZ3Z1vvrSwkNNQZnG3ijsv7Jz/DZ8FKI7iMfWOnkMjCar7Akad/CJNoWxfdmmM5jV9wygOzr9wAB6oEmt56OhnnahjWO4Z/OAlbs8MgYC9bI/buzJrHIQm4VP57QCPPaP1arulPzjQuGLrwEsbuejC875cOEOfVXwDlGRad7Qeu6TpuDmINzhDSsHHlu9kMUilbcOxwhIOIJWhJ7mVAHsrNPKvqwSccXSyC5sgRQ0Uns66l+/xH8vaGE4QcfcAuO9NrXczpp1r2UTqyziej7D3sFUs6EUYRwJLJ79ggehD0FILjmIA/xJvIwtF/5UsdHIHQG7Vo+v3PHJbKb49YT5tUqRufNqek0kUN01E69nk1YeTss4UrKJND1ovZz30KqCPytZ6jF5STV274+Om9kXj4F0N22mlpMkBOwurntTaevWM4XA6m1kFM0eXsq7bhVqU/snyVsNYisJfmYTyfDJ9/7oAFU8DrBiAh3uIVrbxJ4SnB17gLUEe9L72KKlB8HoC2MRnJVnF6IiAocH9zJ7op0936EedHUweH8HJtwvw3rinrgXxGFTtUfXfCRczXKuwjs+Lter7Sbqu+kiWo4TwT46vQMcgxwEvedMUa+Z8r2vB5lnyt1IA+Crp1idYUw70kZnMzrnLsqbrmAjq0zwrTq16nWnbTKTfbXqZwilU+Pil8ZRILfbmbX3OefuXqdLPyYuNkI0S7ljD1x6Hfstax2mEGut8TzkC9ooQvfV1opqfZGK3NeoXgodSFfCz4Ic3J7qsCbZosp7l6+rfm6iAgSa2jei0MA2xwP2v5ydvju7wP2aT2f+u8t/wl9sPDGuovVVH/MGu26kEEfsQ+ScEE/jGcxXS71TjbRNlz9tpRdcSWimpDTTld/EKc9gvc5D2n/PweCxkV5EVLEIY5HCr4XKsRIdz+4Is8DnbCJ6hEaaeqraS6DMtijtKAdww240WSHcCDvZTB9KEJvpR8Rn8oT23e4AxMNpJiLLcBkm6QbPVfegxVMyr2ZVt2JyDe4VbAp/bbmhg+pejNFqpduJMdTfzlqTO5hh1ZkMIQDX8/1mYOweAt2ZZyQgdZUdHijNv5H/1cqAoCwmgcrutpgu+ZikODHORyyi5w3b69jdOYCB76Vf3v7JY/gvb3xwDH9qZwd7elLmIXFOLatLHELvqg6UyZT37411QJ0if/OXzlTG7ry8KrNPMtpTUwIPk/ij0gH3dd1R99EMhdZc0pGmxwxFX63vSKDrEkf7q2ujbfX4hCS6fXL48f2qlAZU/RzwJXXHdppd26oV6k/cBAcGWRanJViu4liofHS2YKibp+u1vK6nWd/JdvgNY4dZzJFjMrFBN3YpXZH6EouTfbSrCkZHdWxL6iaZWfTt+XMx/GaySO3R1tJ5NIWDHfNxA9LHsmg8HGprxOM1Xf60u/J3XAq8YCotBbPRbtHf8i0lvtihUAWa/RJsC0xei4NoA5bSW2EIU0QvAu8I9PYiSFDFo8c2j6Piim62KBBT2DjEuwtFTktf3E64AkIBy6t2/0oCHg8dNKQ7WDAU033LoTEPYWohLuQCza4sWOL24GATFNf4qSxANMj45I/sdhkF6wSYmK+k4cOBBfgE2wQhyTRI7T4+sS2jrP9aNpgtnsS2OqhVb6pZ//r2RssLaDkAz5+3To1o9pl+gISOj1Se3o0NKC+t+/u6DcikqK4TlPtLIK5pUHJGDpThGUy8QKFZWc16vqNRcl0mNauWMFr8WxMvG0PX8k6nlV082z8j6dDUJ6UGJ8YK5OWtfvgL358pjqQSgVHcpUFkvyW/JZ+JeV0Dlob5uHIb758/nxjyIBpt70O9b4pt7zF+6dDtneqiIMmatryhM0pEwpBtFNFmK+4YQicCt0nCZVQWDg7jBzX8I8/8w7hAvvrDeId9wucnEvQ8EXz+yEaegUZe9ZZr88V5YIzJgqAckAoXtw1kzTTjP35L/uj3+/i/2/zzW2KqQCpdNsaE4fJAVz/8YXxbT9mant0b/CTPpYjHhs0sX3faqTUQTzIrsWkDi78lKqNamLNyyoWxTInDKO7ACdaOcfBKKDz7PN8Z48XGkFuTRgkoBvwyWlV6Njnc6BAFQDLFAnqmytWNUrZ2twpdbkecoF1wB68U1dXu3RJ32hnn6s6IQNxIBX2HMuNb7kyjxDbEzXRkpIEspmPVxu0VgGPiefvxw7tzPIx5/uGvtf1uIwDpKxbJkWSgMHsJwxUxMXWNEcaWJM2JLXSN6Vob5o7xK99tUO/WwAgmtQZ4N6iB0GEy2KuXp8uF+zQxWCsg9jbGbbqNl7AOv2+B6/EmyurCQNAyPAq6S5WUiKZB1Nla4/IK142u4l2moUi/56ygUKogZN4gy2CRiu3iClaANYzCEd4hRAKEb2AL8epF03IV6uguRr5ikcUN15gdmS0s/+Uvxgf4bqC+RCEhb9m8vDg9/1CpSiJAkFTKRkHq0GmPQs4h1Da2eAkV0O0qwmmH4ug2Bj+N85UiEsr6hmUFPAA0nLPqiyCvAKuAOJu+4g1JSk1f/PWnwRlFTzi+Y9B2Pr7AYfEWcCjO7muHRwoxxFslQI2LbWKcvyukHSDfO8ZHWOz8lu9AiFJQFNHXsGEB6EMfcEg+l2YB2jIOoDVCMFgT+KMQOAu2YBQABhdIEmDPwD82jKJy5xg/RwlQYIg5MYxlPC6P14wGOyIhsuAUGewzKjZL03JYv1H0F/+o+5FQr1TbLKtobQd3oHfBcHWK7Rzt2KI3sl/YeO4epu31Ri/tl86JtouzCe5AzWw8vIOil9f2Jh7j1AFtD626oq5gzyOxL82bDX3n2BbdHY3g+6i+l0TBFeCqlCBNe8qWBvvRncfgs1l2TzesuSQl4KbV2G7a5PtiWQ/EwjgDvYaIp0ez1KYXrMZ0OHNATJS96VMd5U0+swmH+McO4uwq8JwTGzRBGHsVujz6q52IFJ3iKc87rgyms7yk7FE63rR3oiHaCHheiqWtaU0k9Dhc4xUfVbd32lKPG6MZzZx5kPfu+s7otT1dPx4Pmg0xs50XcvJqPuaejo5kR0+PAu7r8lQp4bpS6RgCYR7EwnXh3dn0hWCAQWySKAbS/fvpe8wcBEUK1q5E9vnjFkcswqhjEVYUbt6kNUYiu3vY5CR6O3rz5jAfAbd46IOBcOhmofpZLctGJrBqIFhO1cEqbI1nDoqp3rSs76ua2lYh3emgmfw2Vm5z4f76jN4KL02OaY4ouLvBpj0CpAkC7XI0AFp4Zr9v6hC1jo5MA2uLU2fMdk1KGTd5VNqT2unCihMxh61hl0pKGCtKWEFTkvSv6zQR3NEdfVDZDe4cALPsCXmisSqesy62YAKUcdhjYqwU+U3R9IbcyjxG209dPWQwsZt1wCVd/yt286wJ9QX2Gnw2NJ+TJWvTXmYRJrAdrrhcYQoUKDvef4UqtbC9UKEcRJpvk6XIzrL5PnNxq2SMjupsgu+86kLzCbjWYJBgxAafHfItiu1qFd31TOdrlDnlJjOt/VePq/a2eWvaqvT8k//u7Of3p5dn7/SLyd3aatHllAXdFMp3rOOoLSdfx+m8Zz43Lcttchh5J05U8PlDi+6adXi8HJnCMQOr07BJb2eep8boIslFiXa1ijr5iBd7Z446XlmmPBigPHGVOtkmvYwvUJkQPqv7nxB3pstnTeROXpXEacFLIFaf9qNNl44PAHE4uHMI7+81ftE7wyGZvJ5+/bA1/XyFs9xusgI6F2m79HFP+XtJ6Y2F8ZMWDl3Cu9CXipZeBB23iU+WvTpxX6Dlmya2SlNTOa9ULohHpMNORG1+Kx7UW2dzvYzwXnx0iArKc7XDOzAS/PRay3pli9/TLqC5+PIBPS+MMIoMO7f2YxC2SbdbgZ+8LRemC3961qEAoB7yfkoksJ6r56qnjrDlUNzlDDoD+QzYH+QK3wUqAn78IAbuoeB2cKA+XXKlbrMugUzVcEWe8VtMNmF/Q/Rq8DU6UoYWjV8/ETGVFH9lII7BWTB4wp1JxyxKRDzG23M71UQPDIrlVfdF1dsIGlUvrdo9Kpx40shCquefeJ2Z1xpSRDLot73UsXfZ5I2lpiuvlLb1o+Gu9ssq9fvBWvlebvXLK10JenoOmsDSky5NatzKLX8TZ/9P31RhNNug8zFVjA0pQjmKZqMPHIwvOzoD8Qg6ruWukmyx1f64JLsBxy4G//j1PadY86U3bDGqACm3pkE2ThQA2fXFL/zk5OaKcEY9e0GGpvW1Jxx6HTmK1dU4zICPv0qtx1eXYCREBVP9Bf9Ui3lU63jqjsYzS/D4I4WcUGA9btTKXHaoiXaDV1em1562VpVOS8Or/+aFzrjdAOw29kAt4PLh78bsyivaAt0VjmA326TfKDLxuLzvi5e+D2JU/JoS6DjxzZFfUD4DuiO8asJ8++kL/mQC4FaEpaTArRa6U9yIUUkBw0945TvFlbbZUsgLjUTqu008/yp5gtQ+v9SCBWKz7Oo42zUl8SlSK0pijKOT5HeMdzl6LUjkIj654DjIJtgZS5hFQgHFT1+IIfbIX7KtsEPn50fe+2CbOf+eC7ypm0nkv4SLVkZk2zWRqXl6Xh5LYpGWVPCVUBP0wj0RqeCd7E1KxlvHtQ68wa1HMNoOiDgCvCf1CT9bbf6CgWjUgyrAyvsXdP6o0MKWeErJuL2CP7QTB4I5FvcwUuQXdCFdSLZ0Wl2gE7g/67FbZeEH5gHwLAbmESGs7XOR3S3ZT6Qo9tiDkpnAKn+Mfq3pwJ7p8+cw0vtWF3hzO9rwUpLw7150iBEy/0BoVGWqa8sVhH4BOEI6VxF35Q6B04IeZhcV75U79MtIVSddmFfWNHyXLD1tmVkzT+0ztamqQ9CxQUdnAllIiM1LerC6UKjg46Fk4CT1/H9GQ88btpmokg7azjXugRl/CFSpYybyTIgIT3mm3XS9HxG9sTtHoB+AMlCHdQDv3FvqXEg9f5txIfOx9buS6ucEhHiwWltbMreepYR2Rvx4ePIgUE2M7Ies3R6jgR+Nhye1rCh9E7OVKhre0c8q/QSS6Iy+oggH8QrvNa9VWPLwsqF1hA9jnv708YKyD+tHkqGB/bdwN0+DfHmO1n6+zUpx07T58+n5e2hSX1ja/iD9BG3J0aVD8fgAShddTdC46scKPfVN/CoWuDILeQfVCiP08c5tIEMOPaSTOOBDeexH3eId3wWaYMuiy03pCxfGbjBnxZo1i4dZsfvHDaTDI46+cNXa4Re5bo37CuqsK9Zu/7p1T1yumcQ/j8KvY95S7pv4xKWAgmd/ouhRe2O1s9liWJ0G3L5DGQ3ge5w2NbCDjR+5di0ks3eilJ3oF/SdjAog8U641hOM53bkSIDD3/5CIFa3bEa1qEWMKO9OhIz+w+N4kbtXpApTsLgChMV7a3ExpTtyxMfm2Q0ymmt7DiJKJ+i9tajkE/lciQb4v0LuIrT0HeT+fSQ/+f9FyEilAuMyuKVgVP3zGx98lCU6zQ28Pwrnh/Ete3gS3if/1sz3s/H/2+UTRuWZ7MA17abA67RDuscuoH3J0KQFUA/zTydwAeYzI9WQC++aj6WOFlT86RKxevp4+VcnRUktck9zf/Y/Swh5dg==')).decode()
assert hashlib.sha256(source.encode()).hexdigest()=='2616d93b908c6e8061842341835a1a82afdca37b3ea8c345127e2bf96b45596d'
(WORKSPACE/'notebook07_coverage_audit.py').write_text(source,encoding='utf8')
shutil.copyfile(WORKSPACE/'notebook07_coverage_audit.py',SESSION/'notebook07_coverage_audit.py')
source=zlib.decompress(base64.b64decode('eNqdGv1P48rxd/4Ki6p1/HBySYBAnfNJ6B6vRboH6MjrUxtZqyXegItju2s7EE70b+/MfnmdD0jf6RTs3dnZ2fmeWR8eHk5oknZn+aJIWcWcivIHVpUOzWKnzGs+Y05BeVKtxs7X29+cPEtXvpPlTpks6pRWSZ45OXcqTpMsyR56h4eHB8miyHnlPNLyMU3u/X+XeeZXyYIdzHm+AGwVDjsK6hZe9YqsXhQrh5ZOVughoGsBW8DYnYHKK3af50/9MzLLl4zTB0ZoHScVQl0cHPz95vvVv26uw8Ho4Pbm29XXf4Y/3HnCy4okWVFXbjD03ZQ2r+6kO3B9N78vGV+KE5HHnCeveSZAC87iZNYeHoz8Awf+ueyFzer23LnvUglO0+QhW7AM95BD5bQKqqPBKBo7ciVzkgyws7IfnLsKZwXyIAWNY+AnrOSsYLQigmLOZjmPWUzUDs9J9ZhkJGaLfO0Ezfosz5hGneZlSYo8TWYrmKmzBS2fABsryiSFRb/eXQLRKRmMiFID4NEsrRETYVlc5EmGRCBBpcapSEnKhrpylhcMNig4yCej2Uwdk1Y5HzvXNxOHGo1xkCYHCdEIs5wvgHWv8iAa1+T7xdU1ubn+9k9y84/L798ubm+vrv9Gfr+6/vnm9ztydf31228/48jl9c+3N1fXE/L98vbyYnKn0ZYVICyrZEYY/C6QFkBb0gVzUItSWhRATvcZKM2fnQWj2aeyiuGhesxjVK1HWA1CntEUhIeaaATG+CLJaEpKtqAZ7FAC4kvFLtBgGMpqaSrwAhzBx7Ej+chioeNobkmJug28gc3LGtTOKfKy6qJ0HcllNLWHmnLYhcH0K+N5d5GLmcc8jXvu28Gvl5OLcOrirjwHkioQLuiGfkc7BE4sxKASnZJ1M2DJshmUzoAofdVnBxEXPMkJWMkyyWtEoodmNeeo/NHBQczmIOInRoCbNKYV7Sh78BG49AKBTI2FWdGjJeWcrjSYNxZw9oxcOJ6EKcsMmERTghlUzuRLODgXbkzN9spHWrAw7Ez8M09MCBzW8OC8hUGvi6sVAkho9QKUzNOcVsdDgQlek3IOCl0xQ0wPTKnjbUxLwuWkvVsHD3dfmuWfw8HRgHVHNqiQpuACqMAD6wz9iS8Jwh2yanTijTn7Tw0iZnEooKeBfw0eIDpqVikHub5S7LAEy4tDg+PzZNyWO4KDsoPzX3QMlA8eVC7nrKp55vxY079A/NmqhXIKxIrEdDRbgZxPw77UMdvLaFUNtD9tUxf1wFusOt5WTQ7E2TZxril20H73N1U8ECKcCsK7g2bPdcW34TTUmzQGtUcJzoTysjMH42XcBy+QzIExPqfP5IHndaEsA/08KhGwJu5gxFQrvE8Wo8GXqLjQy4pX1we1yZ9JkcyeUhb+QtOSeejGXgPDAbGmDF+nrnxyNZVjbYowpR6bOWmKMMPz+7wi6tTNvBAuzm+RtgYSJLRtd6zsQJJiWeXxUBqRmlgzwNHJhoVJQNtuYLZCrw8JityFCCdCQIFpquB9y7kY/jecaaxFK+g+qLWb247bsHYNOZy47crCEEy0WdmjVcXLqQv5EoEABokb4PDCUOvPtK0VMLmL1DSdQfzVLANf0Y/8xlFMhBFCpExD8ERnPsenvjd+/8xC3L6F0ftod6FRuO+MTM26QTAAYprXs2A4iiJDzUhSIzykwP+ySEPLNHzXSkdYxnow73o9zmhMKvZSgaIiP3HVBl8XecxSMk9SpjiXzCE4JxkQA7lMR63z71eoZoGNBv72Ytg3Zm3njvMaQOirSo5RzYenow6uY5lc5/Ue2UucPACzOi2hNumUG4H1ATZxMsRJJB5F7mPy8BhudxfK8TQmCb5itdVXjCHhK3dgkUnSPlj2sBGk1lK6DgaTnwb9o8GmyYE69vdBiaT7O8KkhasloobPcw5JIeTJjJMZPHHqRqFwVUip9EUNMMN8jkiGlAg46Q7XQAqaQDKn8u7GYBF0cCZI0GlR+F6SJCBTusrrKrzozep7RuQbao+/S02wsANFVlbgKmMUq19DY10SERwdSjoVSiSIG0Wo/Xoep2pBu3t7cXfnOgyE7GCCIdCCfvgEidN12QLVxnaCRiDdwel6AuLLLe1UR7NiPaGNxqKiC2HDqSruIFdOqmTJwJSTQshOptUAy5OShR2JvSv/TMEtfQl7/SGeTg7p/BtPs3YuIAMS/xIKSUzGSQLJRMNvPRT5bpVXUAZotx5MfFdLO0D9EbRDotBUEkYbAtQF9Fk4A9IStSkMA5eahEUe1OZEU8y2RwGb/xFfAjFvobfIktsTUIacEzqHCoe0sbmBYP3HcJG1Qcae98GMsQ6UooQUU3DsSyggPA9lJR63i8rayeRw+x2DLqHmpvfpPiuidzmGmVpTCKPiYYL0slIbFXT2ZKS7G3gH03Yjt1iGwyrnlBzDgf+TYR+cYoNdex3E8lhgDuKvFaRlZk/KNEe1x/M0tr8lmY/EWT3Pwr/WoLBRdf77PjLvY2wQFMRCMBsMIHugxBZCG+e2Hk5TymApASlNgmFfGbVCvKAvBEpDCHELy3oFGViO0Ze12hFQ+cEIxL+xuUK46QoEMms5rPZVgi/DAYJbDSbwTOe+q1lAZIvQDWRUeGvFVek7pzvkA6FkMOzLiKkh9/MTsNKs2Nveo62k7WNju7fbvcSuiTWvfIXFF0kKphSyJLyvkzQmy6SEBGazIjSrzSpVGh4eHv5Sp2lX9Woxv70HJyMYCkkJsBp+sspB+iCB5bVqJM0dyFz4yrnE/MWR+Yvo3iJSubudTnvjjwKyrOZkiDsajmEVqFedVU2Wwx/uwx9PwRLQc+fJXwJ9his9KNwWJdTHY5FQlR8DysBMX7AVEfZ7Yxl71eo+kEs5NkEwR+2B8s4lNYyrxLxVUsszftqeqe1bTQti2QqJ/aP9t326ac2GH+fBr1MgKDLKI96mgRGNSgY18ZDsIvF3va8Xv15+v7hr77SNYR1YcuQSEKxgkreLS1I5A5lYh/JN59nKSvQ/ZZpyznQBTPbtn5/g/2OZhiuopiVQg385b+HDg5X1fJ68+GgOL0I8HbfhcEf1cbDm8PyOqxnti3EcXGO5apPVLJTbTwVe0/74SCKGZcQ9koRFvkDXEswYABSoAQsFWIsWqFMVE/aVlTSQvcS1cWq2NLKTaMDsNSPNjOFsNFasNFNNR3iLxNmyl8XJIgxl+QSvQvjTATjeEz1kWrFyD/2uEG82Z7dtZBYr5dKLbV1DL+v5x1LbvG1Y7IYTW+5u9sq9ds2rve1e1do+BsWXsG9j0UvXhzU4FHXD6HM42LZmba61LRZ7yICltyn/5ui4Dv7EyXwOoLLL8yXsYpfIU7ISg9gr7qiZUXsCshbvc4hGjZmPaHW/t6Fd3S9Fe+g48s2T1T+WtaQ6cAOLfPpL8/oZxLrt+OsSTjK5BjbDtGgQbVuEgR/iRsiWyL6d7Ol7MvWGEXGihzLnlZ73n8CNhFhgQ17tetGGE0sw9KP3YlmNTQnUO9TSTTEhNGdzX4RAHzWtfJRuzyhHEiG2bm8g//RPwfNp5dBz/VP8u837yawGE5YqvOvJDkjFMjgOwSOxuKM40iLBGzPOcx5uS1yB3K7C6Hne1g11qMdV6tkXCE3nWLx9xpbgqe9eWbmPzHHWMqA5xbZI4LhHwjGqFAheAaSTeLuCiPDBG0HE2LkdQLRxb+HgfkFCOtn344SE2R4q7Ga7/menSUehMvUxxhG29LXY9HH0EQ7W8hu8ZAzePwPGL5F8KALxuXVHBABqzv/hWpmqpE3XPNhokbIWDU/FEShLHlVvDiDsI9nXO2mq73ZUv7KtAbpaafVj4OTAv7V6xkIq83MirpLUBTUijZGMzUyzq7LQt4ODGdRfpfNrjt8DlFIhMOsnBAMBIZ2SpXP/OYkrSOzxuZdhEosP4ngY0/C6t+wIGN9qZEqoErjOWdissIWPO9E4lpsI1bD9xtK+V5WzLfzKvNrx2Q7PIvE3G29GueW605QHlOq39KxjHoVLUYr3W6c6QgLlI+KSAOZg0rvIswHN4GHEpUBzPn2xJDb90h9jYW4x6pOcgIohlvtw7Z3ELadFh4LsIoKf8MfvW87K3H3iDOgt/K5fax4PPavEhw3dwN6piuUBvPfX1Rko06LAni5igB9rUuieq1RI3TbOQcNaX1V0ZLVa+uIrDJKYMHK/Eh+ThD+41dcMuAwpaPlq4ZtWCpSgWuRJRVAQUgtwoNlDa0pljcHgYNQXm8uLRWUineNRcw2px848Ux6C9QcG1Ntav7yN8ZYFCo4fb2P1PUzYN64Moug9VNji64pWVDW0+QNLiXiojjnFn0ifH/kElRWciUB5J5oSrkDgju0iGsDAmeVpXTESQ405A3+zcsF9SwolzhCgdMHfvtBRiZkkGRjmGPr/POyHYT+AEjGrOvIjmS5+o+W05B24vjqu+wm47YKa1eVjOOE1a/R3a0WMdcofK4eNRHvoeeTl3ut0s8SNfGvUVAiREb5Ybu6hTbEc9TgTHqgDadmZ52kJH4Wy4ff6fqfv/VOrMPPHD/5xOd1OK2TTwCr7d5T7jct+XQ/5jZve/omKcdgB6Ghna2mj//3JuWW8O3ukWQZ5wTJhz6VDl3mCnyoVK/xoizrNB4OcZmWRlwy2EMHREYGktxWzwklMTNNhZ4pZ8Ex2dTeu6iQ7hfsR10fHXuS9i14FQxu/FUHETpsXglIz9t6rSbwiDGWavf3op/MT+D+251Vsa52+BaDjXPsA0iuqD49UmAO7CLVlmCGmHKIZggjYOEW/FA5OFv6t9pkBqGglYMx8oL+Bgri9/NIXkKIPh6DTtRCEveKrDKlaMpVsC0+k5aGS7PYXQuXskS2oGxxDIJTf+MmvNadbPwDEyzXhl9FTYrJlnLQal5/kkXX/GUgHa2Vw2tXIKF1MZbiOmk6bCKriyw9xp6d78AKcNuD66yGEphp6cxvpPIgKy8VUBehorfunAagFsN4w37pAuT1rZ3TZyULcvgqap72T81O/d3I6gp/+CBA3EIL6aW84/KsPPyf4c6oARKpM0Q3qTzDBy4CRX+HCa1aNxYUSpBYVfiVoXZyI9F1JHiQFbk033303Yw9UPWK6TMqaz+lMiFeVkehNQNkW+M1sit3sFUnzh0GBzlgk5QScmZB2hh9SUtlSNYg2CCmA6JS8whk4nEFYic7QykaYMpEyMl/PsDYUHrQF8aqPBORZBZwS3tua7TV2t0Eg4CMi63MDtFp964JBXF29QNYNxbWoWI5HRm9aVgLceu2KM6JYmo9r1cfLLO6+9J1ZmohvXG0mSTV4YDneSkDBMxVhwVjbDLTYDc5GKB5gtXx1ZwzLG3I2QjHi18pKJfWXxdKQ3w7+Byozxz8=')).decode()
assert hashlib.sha256(source.encode()).hexdigest()=='f4a4532058ca130e894cb2ec211b8f4c2442f46fbe624fe600a1bbd57983b145'
(WORKSPACE/'tail_export_math.py').write_text(source,encoding='utf8')
shutil.copyfile(WORKSPACE/'tail_export_math.py',SESSION/'tail_export_math.py')
source=zlib.decompress(base64.b64decode('eNq1XOly20iS/q+nwLh/FNEqQSR1mlxsBNuWuxVjSwrJ7tkNDgMBAkUSIxDA4JBEaRSxD7FPuE+ymXUAhYOS7O5xOCQAVci6Mr88oXfv3l2zrFizvcxdMOPD1Tcjd4PQYA9JnOaWcREbKYt8lgbRkhq/D88Mti5CN49TaiRxGHgbI06NPHWDCHpY79692wnW+KrxjyyOaJzRBHov4nRNs01G82DNKPT22Nz1bmlRBD59DJJFELKdRRqvjcTNV2EwNySRK7hVBKNinWwMNzOiRD3y4vU6jvDZTdkrztk8jm/7J44X37HUXTLHLfwgx14T1QvX6Ig1OmsYAxu/7Oz8fnZ9c355YZPEDVLm7y2TYg+77omue/6w39+7G5CdybeP51+d8482wUcOJ1cO5yWFczdw2DE7YZ479IYnffXGb5Ob32yit5zMj06P+kfDxcExY4vT96eDxcDrv38/OD6CpsXCc98PfJ8tPDbw3h8tDk49f6io/fLt/DNMwfNOj9+/P5ofHh6dHhwPDo5PTg69+fHcOxr0j09c1z84ZUPWP3FPvMHJwBv02fD0aMGOT4bzvsuQ3Nl/XZ19+Hr20X4i/CjJ6In4bB1nZDQ47lOyCjI48cBzQydz10nIoOGojy3l7ckh3qYMOzLfCaKkyOHx8LDfpzsG/NOJiFbHzZ04ddxFzlJnEaRZ7iRxFuTBHSOj40NKInb/lq7D4enp1jHug3zlpMyLUx+mlQYZc5I0ftiQUb82wPZ+hycHlIAMJHEQ5dAlYTCZLIxxeYP3cPzPcvA7Nwx8Nw9iff8Ot2zfYNg/1rZvcIq3re07/sHdG7x9945Ojv6NmzcYHG3dvMNT2LvnnR3gbiOOPOYgZPQQAShsZcHMEZ8XPrARCHiTyZ8FC/7YYg8w56xnjtwsYyDXN5bnRnGEa+hNrDB2/Yqoadq23i7GoOQMaQB4Aeasi9ydh8wQSzH8YLFgaTYyyG6Wp9rwLMzY6MYCGFwHXmvaYkUcdJylm7Me7MU/mJfL9aRxnMv1yOf7ePBFmGf7XhhnsINhHCeOz+5YGCdrFuVkXwEOpwB4vXbTja2vEKnukxKDZBcL24g5hpHy2IvDjldUk+zKB+jYTdUNNlHSnpavOis3W5GZbVcgp5Mp+2e5mxcZdiQfLr9cfT4DyCGGG/lVF0B0EIgc9kDID/QFCav3mRdB6DvZyh0eHVejcjDUh1WzmxIQShf4PlqyNAFFlvOXrI+TrxM+V0696h2BsgJJfuSSXC3Nuri8/lKtbb5x0jhk9lM6JXhBZqPUAD1npEYQVXNdpnGRwDKe+Uu8HTpT0Cdw8MzHvgp8rSBna+Rl3lVwyr0tx5nij1nZgoTi0Kcgg0hi2gON5d1WwNGJOSbtEfcOVBXyuBJd1LqORJ37IPLje3y7eqUcEv/1CDB8jqcDSp1VgzVxC0d6HXro94Faay6ttbw41tvgsDVIbV+3Ax39HvR8fSUvjPMW2DVnChCBhabAKMDAiuWmQEAwUsryIo2MJyWXI3I1ubmBIQR0BT4ZKdihDVkfVZIOzCKYXUkkACNcOWjTvYxK1S6QzIsT0EXkgxJ/o9pOQ70OEwY4ZqApxoYbhkaBdmm4QejO4iL1GOiEDcJbZrgpA7nwQd3AjygPNwjpK4ZHafgFGrNGDiMoM5dILZS5d8yJkkeB5m6aupusWwuN83VicxXEDyFy14w3WHi1Syxo3iO7aN9a+OOwZ1or9iDQNU83lYzj6wb0tmADoh65nxMTjdHFKEosnM6jg5AIAgZ6obegP/8sZ1USiDML1GoIJnUPyNBKSQHgwS5tRki8iMIguu2tgyyDtTvxrf01LTVVylwfzfGOVfPZwUxwV2VzGMb3ThJ4twB+n1xQhHy+j9WKFOqzvPdoIRdkqDPgrjlzBLFbthFaE2FMdChxEMbNGVfNlqDp8A4O+2cBCulxCu/OaioX2CRYbKRdz3mPcruDpVQwiBQWuma5i0qB4g+YGvA6LbUKrbG63IgFCBGQSdCCEHzkSNL1cTSC0olwNF+jrl6TKTcSUdBAPmsz1FqEdgLlCeOuXex5IKUXO9YUupjkPlm7UbCAnetQ6Yr8dlVc9qjGqz2vVoht1d2OfqS6fgPlpBZDty0SFYYixbVpq6feONOgo0dkT7VoBUNNCs12PqbsA2aOj6Y7q1kJTQrdveqTkT6lF/usnEjJWTii7FCH0wbLQTfONLmbLhkqJ/SzCf1iXV1+Pv/w3xq6i5mhIFQA33Xcd0EGMuNoHspsSkCUHblEofn5SiMwdQtP9AHu4CrhzSQ1JQiYtQgenMRNg3zTTQpBQZHjQAGbaYMPmi7nctHKdrEAlEExgUUcgd7raKvsKwRg6gdLOGrkvgb9tpHFgR1UlrsAPljBSUkxokjIHJemcKnTOMIC5/Mh0H0QoYiVCzzho6+AL9bx/TsR9EUU/WJ9Ofs62Z167nqXOOCZFAvYZ750eIRLvrE+TL6cXU9u+EPZjoIISpzdBXGBJptXpCl6FrOZWRu3BcoKK38IlpuUmzOsr1n1em3K7be4V7RyEzj5fJMwQEVgCmcFio0Jv43Wt8vspKAkyO6FoIvVwtEpiXIAHjCTGdryJj09xP8HJvqheNoWwGmaZ3jWPeRe0OHoIhqv0TkQlLpnozgAl1VJN8disUgbToJf9eCiACw6fW0+0G8BjJgfDE3dApQML0RFqFI0e504Kn1XCl0coekkVDVUKpqlvP11lUpT997hjpEUQ1QDW/XfmMNsj3xAtYuG26IA20/4G9LuGxGK3ekiLLKVsGzGfAfgKDHiaCUsXQAmF1xfm7pCV1oBOAZsrDgE+0/QbC+8vt7SH+bUpE0AOCJtEElEjsJn11igvAW0XAYcO+WDhlP9+mtJGGhKTcdXNStbrXIsrRcF+z6eBLgIWa+xHYJMaSbBBYB6RsGazugqWK7sL5YkkQHWu2mLgHbG+owkIRCFYs7AidzERc4XUZkj0ge5CqKoPGADuxuiu0JaS6gSaXjZiv/2ldklFG1G9nHz98vAgrCJyiCSeLuKI5WSuGQROB2o16Rc2HXrsuT3P25hNi3n0t6Qfg9ui9y4SmsJobhmBRr0wvINYL+04H2XVKTQHxgCb6pAVhVvWM6pULFUaHc4ZjFpcds+5XKZJWeYHTv46uFUXeFR78Yqcg8cJuXWgF9ICdHud+HeISbH86aLNR2dzrrmYK1v/QB1AaqQjO8A5aeOzpDQwmOuZu2n55qHso59MHHyjXSMpEWLoErhBzfp+JYRuXVmQz1x+6Kax35PEYTpdxgygFxdTui400kTM56WjufMbtop5VR+Mr4Ga2CVzLgDD9yYs/yesQjsC/AIYdu5WkGv2WdhMMe5MnCa2YMXFj6wFc/NSFY2AnSpYQFWSVywh5XESa9tFzoZWpV+RnARXJK0iEPlfFDlcYwOqO5kjDQJqm1sqSJGnM3rLsLoRf+BbnMaRq85DY0pvOg/jN7mP9Aup2FUgUV9xE7/YVTXr13ew0h5D7TClZHC9Q5jfiQeUWk4j/ivSjKqeL3G2g3HUwJnxYONiLkEzte5AdQ+nHKXP72VUxrMUd/DlIUuRvscQARQ8XEK24MBfg0tyi55XEK9adIWQ+jBru07YT5Xm/BvUyMK3DmYqRCB/fTHxETlp0rFrZ5okkDcOVhORf7Cfpr0D2x6NdAP7j79c+S0mgc6qGoKDQdzus17nZXu69Y3t7u3Mn8g7XV4kT79/HMpvfUjhdb2QQrewEb8DRsiMXnUYR/vSdtZBkXd5TJlPI+FlqI0kURew57OfiSvkbFQ+FlTLWWCxNEkS5uTt22e+xjzITE6IFaJw6kkK7pZiqj53AwwInU1sbpuhuGg/S+2osOHEOGUrFj3Un6pZXXUEE2LDVz6nvaufVe50frgaieAIb+inVYGtv2YZVi5YKzd3FuBlexmReXeGDI9EKDVgAEGXHtl4vBxeTBHIr5Ir85s2VJK7M+D4+ZbzczsInVV6GdLD0l8f+uwzSHW7oMDAAHGfoaRgjXGs0E+MXoITbDHRLTwcR3Vu9bxpRMQjGi5CYb6e5rHIWVFtAtOXgRL2NasJ40eiiwnlVRGeeED4KTicFkpAieSwMLAKBpXlxYIUY9MlkuwaVr9rGSDVxjTScJcl5aGJIH1tqTuA5ic0M/Kijm+lvUG9IBCSxY8Mrs3GNJD0yyFzH2gyFVgfYU8OPMYJD2kQKtjpm/KvL2WQdLQbkomIfIqgCIeEc+Uq8TfOb8w3Hyfj2PwcQw1jqFOsOzIo2FqPOHV4aDGf9pDbw3eayWe4GjASm09okh24ckYFAM2cLnm4gs+o1q7iHLwFBnvXLlLfEZg52LMxE3RgewNzbH7YM1dcApU6541OKXT5VQMLvgOVysPcEatg2MKkshCm/xWJah4RGIFN49luJ2DQ4P6rqQuVvAydYkPwmclfKaIiA954N1mFUk496/Xk/ML2OHfJ5/PP06+nl9ekFltDvgeZ5ge/0kXMSgZZK73JdkNH7VHPuiHrLLBvFfIlihf5bsnKse0BMZNBHnyEXBbz9ahwTbmuOYaIXNTcOj3ZNFYVngeSKEhCh8IlwUrB/8xl1GBnjYAOETwW4ntPqmVW1lJtARPNwnswXFf0HntBX+BJREgc7zcogddxWClXorYQ97TdZPACKWeqqRNCRrT/gzG5t6xyBIqWtMu62hWxleAgdqjaZpQyw+VBKusyTjnGDqUARo0JXQ5d5ZpALiYakgKYwDfB3a+N2wn+FT+aFv8fWvkGl6wnzx3PXqUYekyVDsNZp2B3+fto2+3grZPgL/zQ1PoxmBAXQ2DB/RIw2A4HippsahYc1+5V1LVIAzJTrF3f2YF62wV3/dgo3B6s3oSFCNaEQv52XNoOjBH5cuycXdQEuGL5WSmsnFGvbWb2KBrl+AyIe/Ihv8YSki8C9LADzLSGDgOBeDUFzMl17/+ApASxstBYlxeVJefPsE1mowgwOnC9TAXp001Dmca3PBzMP4FBgsfowKeU4QU/R33Ich6JF4sSBescITbi6NwU+b5MSnA7nlhVlss+KAy1m5grF3Wb4Ev8f0ww1WdIwfUsOao34IQkUwvIuXbqQiyMjeqPKAsKRDt9VqwseytUITflE9fjGKJlL5eIKa51tffLi7OL36F4wMUzLiHL6tdab2gSos8EOkIOEXuoa/Fw3JodN8FHq/WuPpGNK9IVQE76E7E6Rr9DS6klEhhRjnXn642GTgwsF/MA4CsGkqHTDp1WlOzQGzUf9bS4eiRjBE/8fcCzg/sPfsijtir+YBaUQY61tJDkeeCAeQIEWEfHni8GkMG0TOyj6G3Mvco01YiVrFMCkdUCKu9wTieLIChrV5YVaW81tmsPR1nzmAYZj/xckRTd38Tk88gweHL/pUrxEt67I6axHE9KFMyf1mzUxbdyElzGqZup1V5H9rIYcj7jPIkhT2xlGdXm0JtnbxK8el7mFRGxLR4kRAPRw8NiXK/mzNezFSPBmm962EBrUSwpNlRHDjSSgNpdxYHSeH53lx+u/5w5lxdX369/HD5WbzSiE259+UiJ9b15G/OzW8TWs/ylOSuPp9/Fe31WnfnhWqtVjywM6KpRwLJ5PNnhbzIawYSBYd1F/gdzYw9TCMJq2XXwEuREwKvOFhGWL4qyrXOUG0pq7JdnaUKD8AviCPpSIijHWuVuWWiQzQZIp6Bb6T8Mwby3OImvgd2Zy3ruGIRWcmvxxjvBuBF1KhMR4PhTCvPjHO7xAckVYHDvibaWsASMx7lkGMk8EZEr8dcuyp3q4zkNpHe1r+Ky8jyvCLB6FdP2xwt/FgFZ2D1iEM8SlePRdpbElxbJgY6k8OymNcTSTb5Chkv22SWBAJK+OcfBKsPHEc+dBxgZ/l5CRmpK0tdtHUVweNI2T8LPBupUrTQrHTlV0fJpploE1Ue6Ba5VfaZJ9uEL2wZNzJTiflV2MjMWLsbw4fVgpAw42OK/jCuEh0i49erb0aQGci7FtEzdPW6EZyJ9YnDe43NQMj2w2ABcg7eu3tvrfzFEZbhitpBeDJqVV7AOmLuXYhSBFld3a6kEHIus/HZlL+m2VaoXcH2lp+J8BRqowcP0yBaTUUnVcitDNEKWHa2VTzwsdthQDVSR3SwRQpDpDI+am+vZehWV/Q76hjEdPYRJxobZXbNCU0UFauCW5PbKuqBnC8Ye9ur4THSKem0B9gm9xqmqVXVKnGpvGtTFNyvD0rJPhbk/4tnd7T1itsqOyAjTGCJq+JwQsskQrNefNYpApId9NFtvgFPNX5reunPJv9moBEhbAXfaprc/mItgoZ2V4PSLm5+AZXFbtdIyW2uPRMzaCuneqdyGD6uPX26HcngsggtpyqmzINhf7G7og3PrS2qdJjK5YLJ1cgstpKKL9pZ2y2nds4WjPZX1sFjRtyO7lhOZypp1rHIxtC1jQVFonZfGlegbvCXlj2oVlQ7ILVl2tkIofrxvLZimvZOajeYii1vnLmbgTs/UpP5kSxx01RtWpddxup32tZd5rK28fVmtC2rNG+XADVe2PrBwXYSZtPc1Y6+TCfxX1QKfJkCFTgy7c9qMKfXOTR6Dmad6VI+rvhyVa+SkGNLsNnWPnstuY8cyNADd+I5wOed5PbBqWrK49wF4T44brpBsGOBSMKoUPZocEyJRqZqGFIiEzZ+AKbXCTogPnPaXtv3lDZooX4uCKj9YNw5coX0/jGyz3NkqM8ql1oZdKg/nl9DZsm6kqXkXT0uxmuIywpY/nXVC6/S3ttBH/qWrppMXYlu4sacjbqVeDUj3Zq+TwE1HS+7e1nZS71rQUfCk15m07y9KuZhkK3Qvi3j9zJbxiuDeAhOhd0sq26zjl/Mq72sPavQBk8gvT2yMda+8qvFRuwGSapnatDrfGhUEL5gOdW+6YI3HazRWGJBe+mrNL+dErMQWKvPCwQGZ6M38Aea+8FDQ/brAaf9VnCJI0JHhKlS8zj7P6CfSo2LkiTqNrjzqldsqHBOE2JLLnyheGO7gJpvVY4vq5o/vYjqu5DoJ+OacfffWBeYHwXeNQRbgJd4K75GYwZ+zKlq7O6CDIHPyGNjUeQgXdXfWOjCNzxyyg95qzcvjl7/3Edijsj0Z3TLNtv6Lrc32d6yx3V2FOIjwgV4byrXS5i1wMS0fEE8wr870NR+9mG/YxCld+1OBd3Wz3anem6ftl0edh0GAhBBp4gkiIiITW1WmM/jEU6bfEsw5G10gfK8iPyQWY9BYgF3IK4aoFILTCdXG16yycXZ3wzJbsaF+hMXh/v9I8v4GHNzOS0i/aNJtZ3lH8TIMEgGbAZOuIxFWTLbwh48luTGLzCtM34ZiL+pAc+r+IAKpsPD8RbOmvxyec0/IwMTHtg3gufAoT14g/6Vbeaxm/rnGHFPiySXHySQT5Pzz8iK8DROOX9Ad7P6EyF2eWVhSMfNHWjvmfUvHLexPIYXefLCVrkLgKuEI6kof7JfqH56NWr1drd6/Gdo6rcNvPaJKQfLMa1MfqrY5cD4v//5XwNte71M/O/R3yOyixO2/GKdZL2ma6ZINxzN8mDIM0ceQIoh9dnCLUL0l5CgI2/NXQJjUBaBGQgsaZMiX5xqqciJJYShNCBeXGolOKRlx9xwVhyVW199WUBhCkIasfWN9FthCVi7FAQM36HYYXJpBLIGzCxb9OojOY2d/wfXWpj/')).decode()
assert hashlib.sha256(source.encode()).hexdigest()=='2cbd179868504311e49c9b6f85c17a77ab4d55a775a89e6aad58383122d3024c'
(WORKSPACE/'tail_export_controller.py').write_text(source,encoding='utf8')
shutil.copyfile(WORKSPACE/'tail_export_controller.py',SESSION/'tail_export_controller.py')
# Avoid accidentally using implementation modules from an earlier interactive session.
for name in ['common','contracts','notebook07_coverage_audit','tail_export_math','tail_export_controller']:
    sys.modules.pop(name,None)
from tail_export_controller import run_export


## 3. Automatic export, resume, normalization and evidence

Run All creates a **new schema3 namespace** under `data/processed/paired_gpu_tail_exports/d200_paired_tail_v1_<hash>/`. Historical source files and schema2 exports are read-only. The old training-ready index is protected; the new index is `configs/processed_sources/paired_gpu_d200_tail_training_ready.json`.

Expected totals are7460 TRAIN and1806 VALIDATION inputs:2400+600 restored windows. Each demo has120 endpoint-repeat target slots, recorded explicitly. These targets receive ordinary training loss in the planned protocol; do not silently mask them.

Every rerun rechecks source hashes and raw alignment. Completed new exports are reused after output checksums and complete target/metadata checks pass; Event reconstruction is reused only when source and exporter identities are unchanged. Drive verification still takes time. One camera at a time and sorted timestamp searches reduce CPU/memory overhead. GPU compute is unnecessary here; no measured wall-time speedup is claimed.

Only after200 exports, TRAIN statistics and required report/preview gates pass is the new training index published. Interruption or failure preserves per-demo completed generations and session evidence. COMPLETED describes dataset export, not training or Lift success.


In [ ]:
EXECUTION=run_export(PROJECT_ROOT,SESSION,BUILD_SHA)
print('Execution status:',EXECUTION['status'])


## 4. Review, upload and next gate

Upload **`paired_tail_dataset_bundle.zip`** from the exact printed session directory:
`MyDrive/DiffusionPolicyEvents/results/paired_source_preparation/notebook03_tail/sessions/<session>/`.

The bundle includes schema3 dataset identity, normalization, coverage CSV/JSON, PNG/PDF coverage figures, TRAIN preview, historical-index protection, runtime and hashed implementation records. It excludes source arrays, new NPZ exports and model weights. The same bundle is written on a controlled failure; upload it then too.

**Stop here. Do not run historical Notebook04/05 against this new dataset.** Their loaders target schema2. Actual dataset and normalization fingerprints must first be reviewed, then new matched RGB/Event training notebooks and new Run IDs can be prepared with the explicit endpoint policy. New training must initialize afresh; old checkpoints cannot resume under new statistics.

No change to visual encoders, U-Net capacity, losses or training budget is executed here. Subsequent learned-policy and paired closed-loop development tests will determine whether the coverage repair helps. Final evaluation remains protected.

Local tests cover numerical targets, stored-array parity, integrity and resume logic using fixtures. Real Drive/HDF5 and Colab execution remain the required validation gate. After downloading evidence, disconnect and delete the Colab runtime.


In [ ]:
from IPython.display import display,Markdown,Image
display(Markdown((SESSION/'paired_tail_dataset_summary.md').read_text()))
if (SESSION/'tail_coverage.png').is_file():display(Image(filename=str(SESSION/'tail_coverage.png')))
print('Upload:',SESSION/'paired_tail_dataset_bundle.zip')
